# Train with PPO + Command Conditioning

**Retrains from scratch, not warm-started** - unlike stage 04 (same observation space as
stage 02, so warm-startable), this env adds `cmd_vel` to the observation, changing the
actor/critic's input size. A stage-04 checkpoint's weights can't load into this shape (first
layer mismatch) - same reason the reference repo's own `train_with_ppo_cmd.ipynb` retrains
from scratch instead of warm-starting.

**No yaw command** - only forward/back velocity is commandable. See `uniwheel_env_cmd.py`'s
module docstring: neither actuator has a torque component about the yaw axis, so there's
nothing to condition a yaw command on (same reasoning as `yaw_penalty_coef=0.0` in stage 04).

Phases (condensed from the reference repo's 10 down to 10 still, just without its
yaw-specific split - we don't have a yaw axis to introduce separately):
1. Balance only
2. Add position penalty + pitch/roll-rate penalties
3. Small velocity commands
4. Full velocity range
5. Mid-episode command resampling
6. Observation noise + action delay
7. Motor noise + random pushes
8. Mass + friction randomization
9. Motor gain randomization
10. Ridge torque (wheel axle only - see module docstring for why)


In [1]:
# Import standard libraries
from dataclasses import replace
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
import numpy as np
import torch

from uniwheel_env_cmd import UniWheelEnv, DomainRandomConfig

sys.path.append("..")
from rl.ppo_trainer import PPOConfig, Agent, TrainResult, evaluate, train, \
    export_tb_plots_as_csv, export_actor_onnx


In [2]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")
SEED = 42
NUM_ENVS = 12
STEPS_PER_ENV = 500_000


In [3]:
ppo_config = PPOConfig(
    exp_name = "uniwheel-ppo-cmd",
    env_id = "UniWheel-v0",
    seed = SEED,
    cuda = False,   # see stage 02's README - measured slower here, not a typo
    num_envs = NUM_ENVS,
    actor_hidden_layers = 2,
    actor_hidden_size = 48,
    critic_hidden_layers = 2,
    critic_hidden_size = 48,
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,
    num_steps = 2048,
    num_minibatches = 32,
    update_epochs = 10,
    anneal_lr = True,
    learning_rate = 3e-4,
    gamma = 0.99,
    gae_lambda = 0.95,
    clip_coef = 0.2,
    value_clip = 1.0,
    ent_coef = 0.0,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    checkpoint_interval = 20,
    save_model = True,
)


In [4]:
def make_uniwheel_env(render, debug, **kwargs):
    """Create one uni wheel command-conditioning environment."""
    env = UniWheelEnv(
        mjcf_path=MJCF_PATH,
        render_mode="human" if render else None,
        debug=debug,
        **kwargs,
    )
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs uni wheel environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i == 0), debug=(i == 0), kw=kwargs:
                make_uniwheel_env(render, debug=debug, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

# Fix a quirk in the shared evaluate(): RecordEpisodeStatistics doesn't forward custom
# attributes, so evaluate() falls back to config.num_steps (2048) instead of the real
# max_steps (10000) - see rl/README.md. Apply this right after building any env set below.
def fix_eval_max_steps(envs):
    for w in envs.envs:
        w.max_steps = w.env.max_steps

eval_config = replace(ppo_config, timestep=0.005)


In [5]:
def load_agent(envs, model_path):
    """For debugging only! Load a previously saved model to skip earlier phases."""
    model_path = Path(model_path)
    device = torch.device("cuda" if (torch.cuda.is_available() and ppo_config.cuda) else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    return TrainResult(
        agent=agent, checkpoint_dir=model_path.parent, best_model_path=model_path,
        final_model_path=None, best_mean_return=0.0,
    )


## Phase 1: Balance only

Same starting point as stage 02 - only penalize tipping, no velocity tracking yet
(`vel_reward_coef=0.0`).

In [6]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-01"

envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    roll_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    vel_reward_coef=0.0,
)
fix_eval_max_steps(envs)


In [7]:
result = train(ppo_config, envs=envs)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-01


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593/checkpoint_iter0020.cleanrl_model
New best model saved (mean_return=182.81) to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593/best_model.cleanrl_model


Checkpoint saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593/checkpoint_iter0040.cleanrl_model
New best model saved (mean_return=406.50) to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593/best_model.cleanrl_model


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.093 m/s vel_tracking_reward=0.0000
  pitch=-0.193 roll=-0.003


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.341 m/s vel_tracking_reward=0.0000
  pitch=0.008 roll=-0.001


...

--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.047 m/s vel_tracking_reward=0.0000
  pitch=-0.005 roll=-0.001


--- step 5000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.039 m/s vel_tracking_reward=0.0000
  pitch=-0.006 roll=-0.001


--- step 6000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.043 m/s vel_tracking_reward=0.0000
  pitch=-0.001 roll=-0.001


--- step 7000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.044 m/s vel_tracking_reward=0.0000
  pitch=-0.002 roll=-0.001


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593/uniwheel-ppo-cmd-phase-01_final.cleanrl_model
Best model mean return: 9985.45, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__42__1788710593/best_model.cleanrl_model
Best mean return: 9985.45


In [8]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.011 m/s vel_tracking_reward=0.0000
  pitch=-0.002 roll=-0.001


--- step 2000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.013 m/s vel_tracking_reward=0.0000
  pitch=-0.004 roll=-0.001


--- step 3000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.011 m/s vel_tracking_reward=0.0000
  pitch=-0.001 roll=-0.001


--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.014 m/s vel_tracking_reward=0.0000
  pitch=-0.002 roll=-0.001


--- step 5000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.010 m/s vel_tracking_reward=0.0000
  pitch=-0.007 roll=-0.001


...

--- step 7000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.024 m/s vel_tracking_reward=0.0000
  pitch=0.004 roll=-0.001


--- step 8000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.017 m/s vel_tracking_reward=0.0000
  pitch=-0.003 roll=-0.001


--- step 9000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.008 m/s vel_tracking_reward=0.0000
  pitch=-0.003 roll=-0.001


--- step 10000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.007 m/s vel_tracking_reward=0.0000
  pitch=-0.004 roll=-0.001
Mean return: 9985.09


Exported charts_metrics.csv (5 metrics, 6621 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 2: Position + rate penalties

Introduce the position-holding penalty (only active while cmd_vel=0 - see the env's
`step()`) and pitch/roll-rate penalties for smoother motion, still with no velocity
command yet.

In [9]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-02"

for w in envs.envs:
    env = w.env
    env.pitch_penalty_coef = 5.0
    env.roll_penalty_coef = 5.0
    env.position_penalty_coef = 0.01
    env.pitch_rate_penalty_coef = 0.02
    env.roll_rate_penalty_coef = 0.02
    env.vel_reward_coef = 1.0
    env.sigma_vel = 0.1
    env.vel_max = 0.5


In [10]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-02__42__1788715361
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-02


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.003 m/s vel_tracking_reward=0.9999
  pitch=-0.001 roll=-0.001


--- step 2000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.014 m/s vel_tracking_reward=0.9981
  pitch=0.001 roll=-0.001


--- step 3000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.017 m/s vel_tracking_reward=0.9971
  pitch=-0.003 roll=-0.001


--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.024 m/s vel_tracking_reward=0.9945
  pitch=-0.008 roll=-0.001


...

--- step 3000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.000 m/s vel_tracking_reward=1.0000
  pitch=0.001 roll=-0.001


--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.002 m/s vel_tracking_reward=0.9999
  pitch=-0.003 roll=-0.001


--- step 5000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.002 m/s vel_tracking_reward=1.0000
  pitch=-0.001 roll=-0.001


--- step 6000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.006 m/s vel_tracking_reward=0.9996
  pitch=0.001 roll=-0.001


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-02__42__1788715361/uniwheel-ppo-cmd-phase-02_final.cleanrl_model
Best model mean return: 19975.27, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-02__42__1788715361/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-02__42__1788715361/best_model.cleanrl_model
Best mean return: 19975.27


In [11]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.009 m/s vel_tracking_reward=0.9992
  pitch=0.006 roll=-0.001


--- step 2000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.007 m/s vel_tracking_reward=0.9995
  pitch=-0.003 roll=-0.001


--- step 3000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.003 m/s vel_tracking_reward=0.9999
  pitch=0.002 roll=-0.001


--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.002 m/s vel_tracking_reward=1.0000
  pitch=0.002 roll=-0.001


--- step 5000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.007 m/s vel_tracking_reward=0.9995
  pitch=0.003 roll=-0.001


...

--- step 7000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.008 m/s vel_tracking_reward=0.9994
  pitch=-0.004 roll=-0.001


--- step 8000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.005 m/s vel_tracking_reward=0.9997
  pitch=-0.004 roll=-0.001


--- step 9000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.002 m/s vel_tracking_reward=0.9999
  pitch=-0.005 roll=-0.001


--- step 10000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.000 m/s vel_tracking_reward=1.0000
  pitch=-0.001 roll=-0.001
Mean return: 19977.92


Exported charts_metrics.csv (5 metrics, 651 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 3: Small velocity commands

First real velocity commands, kept slow (`cmd_vel_range=(-0.5, 0.5)`), with a 50% chance
of commanding stillness each episode so it doesn't forget how to just balance.

In [12]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-03"

dr = DomainRandomConfig(
    cmd_vel_range=(-0.5, 0.5),
    cmd_zero_prob=0.5,
)
for w in envs.envs:
    w.env.dr = dr

# Encourage more exploration and allow bigger value updates - helps avoid getting stuck
# ignoring the new command entirely (a real local optimum: "just balance, ignore cmd_vel")
ppo_config.ent_coef = 0.01
ppo_config.value_clip = 10.0


In [13]:
# DEBUG: uncomment to skip phases 1-2 and load a previously saved model instead
# result = load_agent(envs, "runs/UniWheel-v0__uniwheel-ppo-cmd-phase-01__.../best_model.cleanrl_model")


In [14]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-03__42__1788720076
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-03
New episode: cmd_vel=-0.061


--- step 1000 ---
  cmd_vel=-0.061 vel_target=-0.031 m/s vel_actual=0.009 m/s vel_tracking_reward=0.9849
  pitch=-0.003 roll=-0.001


--- step 2000 ---
  cmd_vel=-0.061 vel_target=-0.031 m/s vel_actual=-0.001 m/s vel_tracking_reward=0.9911
  pitch=0.001 roll=-0.001


--- step 3000 ---
  cmd_vel=-0.061 vel_target=-0.031 m/s vel_actual=-0.003 m/s vel_tracking_reward=0.9926
  pitch=-0.003 roll=-0.001


--- step 4000 ---
  cmd_vel=-0.061 vel_target=-0.031 m/s vel_actual=-0.012 m/s vel_tracking_reward=0.9966
  pitch=-0.008 roll=-0.001


...

--- step 6000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.020 m/s vel_tracking_reward=0.9961
  pitch=-0.002 roll=-0.001


--- step 7000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.031 m/s vel_tracking_reward=0.9907
  pitch=-0.000 roll=-0.001


--- step 8000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.028 m/s vel_tracking_reward=0.9920
  pitch=-0.001 roll=-0.001


--- step 9000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.022 m/s vel_tracking_reward=0.9950
  pitch=-0.002 roll=-0.001


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-03__42__1788720076/uniwheel-ppo-cmd-phase-03_final.cleanrl_model
Best model mean return: 19958.47, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-03__42__1788720076/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-03__42__1788720076/best_model.cleanrl_model
Best mean return: 19958.47


In [15]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=-0.335


--- step 1000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.122 m/s vel_tracking_reward=0.9790
  pitch=0.008 roll=-0.001


--- step 2000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.109 m/s vel_tracking_reward=0.9658
  pitch=-0.000 roll=-0.001


--- step 3000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.116 m/s vel_tracking_reward=0.9735
  pitch=0.001 roll=-0.001


--- step 4000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.105 m/s vel_tracking_reward=0.9610
  pitch=-0.003 roll=-0.001


...

--- step 7000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.108 m/s vel_tracking_reward=0.9654
  pitch=-0.003 roll=-0.001


--- step 8000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.106 m/s vel_tracking_reward=0.9623
  pitch=-0.007 roll=-0.001


--- step 9000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.108 m/s vel_tracking_reward=0.9654
  pitch=-0.005 roll=-0.001


--- step 10000 ---
  cmd_vel=-0.335 vel_target=-0.168 m/s vel_actual=-0.113 m/s vel_tracking_reward=0.9701
  pitch=-0.002 roll=-0.001
Mean return: 19914.06


Exported charts_metrics.csv (5 metrics, 293 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 4: Full velocity range

In [16]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-04"

dr.cmd_vel_range = (-1.0, 1.0)
for w in envs.envs:
    w.env.dr = dr


In [17]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-04__42__1788724836
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-04
New episode: cmd_vel=-0.122


--- step 1000 ---
  cmd_vel=-0.122 vel_target=-0.061 m/s vel_actual=-0.048 m/s vel_tracking_reward=0.9982
  pitch=-0.003 roll=-0.001


--- step 2000 ---
  cmd_vel=-0.122 vel_target=-0.061 m/s vel_actual=-0.055 m/s vel_tracking_reward=0.9997
  pitch=0.001 roll=-0.001


--- step 3000 ---
  cmd_vel=-0.122 vel_target=-0.061 m/s vel_actual=-0.055 m/s vel_tracking_reward=0.9996
  pitch=-0.002 roll=-0.001


--- step 4000 ---
  cmd_vel=-0.122 vel_target=-0.061 m/s vel_actual=-0.060 m/s vel_tracking_reward=1.0000
  pitch=-0.006 roll=-0.001


...

--- step 5000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.023 m/s vel_tracking_reward=0.9946
  pitch=-0.001 roll=-0.001


--- step 6000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.027 m/s vel_tracking_reward=0.9928
  pitch=-0.004 roll=-0.001


--- step 7000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.026 m/s vel_tracking_reward=0.9934
  pitch=-0.002 roll=-0.001


--- step 8000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.027 m/s vel_tracking_reward=0.9927
  pitch=-0.001 roll=-0.001


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-04__42__1788724836/uniwheel-ppo-cmd-phase-04_final.cleanrl_model
Best model mean return: 19855.32, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-04__42__1788724836/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-04__42__1788724836/best_model.cleanrl_model
Best mean return: 19855.32


In [18]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=0.000


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.026 m/s vel_tracking_reward=0.9934
  pitch=0.001 roll=-0.001


--- step 2000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.028 m/s vel_tracking_reward=0.9920
  pitch=-0.002 roll=-0.001


--- step 3000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.024 m/s vel_tracking_reward=0.9945
  pitch=0.001 roll=-0.001


--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.028 m/s vel_tracking_reward=0.9922
  pitch=-0.001 roll=-0.001


...

--- step 7000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.028 m/s vel_tracking_reward=0.9922
  pitch=-0.001 roll=-0.001


--- step 8000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.031 m/s vel_tracking_reward=0.9902
  pitch=-0.003 roll=-0.001


--- step 9000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.029 m/s vel_tracking_reward=0.9917
  pitch=-0.003 roll=-0.001


--- step 10000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.024 m/s vel_tracking_reward=0.9942
  pitch=-0.002 roll=-0.001
Mean return: 16444.65


Exported charts_metrics.csv (5 metrics, 926 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 5: Mid-episode command resampling

The commanded velocity can now change *during* an episode, not just at reset - closer to
how a real operator would drive it.

In [19]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-05"

dr.cmd_resample_prob = 0.005   # 0.5% chance of a new command each step
for w in envs.envs:
    w.env.dr = dr


In [20]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-05__42__1788729622
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-05
New episode: cmd_vel=-0.122


New command: cmd_vel=0.355


New command: cmd_vel=0.000
New command: cmd_vel=-0.021


New command: cmd_vel=0.238


New command: cmd_vel=0.000


...

--- step 6000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.054 m/s vel_tracking_reward=0.9709
  pitch=-0.006 roll=-0.001


New command: cmd_vel=-0.842


New command: cmd_vel=0.000


New command: cmd_vel=0.000


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-05__42__1788729622/uniwheel-ppo-cmd-phase-05_final.cleanrl_model
Best model mean return: 18727.65, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-05__42__1788729622/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-05__42__1788729622/best_model.cleanrl_model
Best mean return: 18727.65


In [21]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=0.000


New command: cmd_vel=0.000


New command: cmd_vel=0.000


New command: cmd_vel=0.000


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.017 m/s vel_tracking_reward=0.9971
  pitch=0.009 roll=-0.001


...

New command: cmd_vel=0.000


New command: cmd_vel=-0.946


New command: cmd_vel=0.000


Mean return: 5965.65


Exported charts_metrics.csv (5 metrics, 1309 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 6: Observation noise + action delay

Same as stage 04's phase 2, layered on top of command tracking.

In [22]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-06"

dr.pitch_noise_std_dev = 0.01
dr.roll_noise_std_dev = 0.01
dr.pitch_rate_noise_std_dev = 0.01
dr.roll_rate_noise_std_dev = 0.01
dr.wheel_vel_noise_std_dev = 0.1
dr.action_delay_steps = 1
dr.action_delay_random = True
for w in envs.envs:
    w.env.dr = dr
    w.env.smoothness_penalty_coef = 0.2   # discourage jittery response to noisy obs


In [23]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-06__42__1788734400
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-06
New episode: cmd_vel=-0.122


New command: cmd_vel=0.238


New command: cmd_vel=0.898


New command: cmd_vel=0.000


New command: cmd_vel=0.000


...

New command: cmd_vel=0.000


New command: cmd_vel=0.000


New command: cmd_vel=-0.112


--- step 3000 ---
  cmd_vel=-0.112 vel_target=-0.056 m/s vel_actual=-0.035 m/s vel_tracking_reward=0.9956
  pitch=-0.009 roll=-0.007


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-06__42__1788734400/uniwheel-ppo-cmd-phase-06_final.cleanrl_model
Best model mean return: 17011.63, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-06__42__1788734400/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-06__42__1788734400/best_model.cleanrl_model
Best mean return: 17011.63


In [24]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=0.000
New command: cmd_vel=-0.799


New command: cmd_vel=0.000


New command: cmd_vel=0.000


New command: cmd_vel=0.386


New command: cmd_vel=-0.846
--- step 1000 ---
  cmd_vel=-0.846 vel_target=-0.423 m/s vel_actual=0.139 m/s vel_tracking_reward=0.0423
  pitch=-0.011 roll=-0.002


...

New command: cmd_vel=0.000


New command: cmd_vel=0.000


New command: cmd_vel=0.432


--- step 10000 ---
  cmd_vel=0.432 vel_target=0.216 m/s vel_actual=0.071 m/s vel_tracking_reward=0.8103
  pitch=0.029 roll=0.021
Mean return: 15661.70


Exported charts_metrics.csv (5 metrics, 902 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 7: Motor noise + random pushes

In [25]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-07"

dr.motor_noise_scale = 0.02
dr.push_prob = 0.005
dr.push_force_max_n = 0.3
for w in envs.envs:
    w.env.dr = dr


In [26]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-07__42__1788739265
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-07
New episode: cmd_vel=-0.122


New command: cmd_vel=0.000


New command: cmd_vel=0.000


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.014 m/s vel_tracking_reward=0.9980
  pitch=-0.015 roll=-0.011
New command: cmd_vel=0.000


New command: cmd_vel=-0.388


...

New command: cmd_vel=-0.703


New command: cmd_vel=0.000


--- step 10000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.059 m/s vel_tracking_reward=0.9659
  pitch=-0.009 roll=-0.005
New episode: cmd_vel=-0.586


New command: cmd_vel=0.000


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-07__42__1788739265/uniwheel-ppo-cmd-phase-07_final.cleanrl_model
Best model mean return: 16570.39, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-07__42__1788739265/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-07__42__1788739265/best_model.cleanrl_model
Best mean return: 16570.39


In [27]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=-0.537


New command: cmd_vel=0.000


New command: cmd_vel=0.000
--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.008 m/s vel_tracking_reward=0.9993
  pitch=0.009 roll=-0.001


New command: cmd_vel=0.482


New command: cmd_vel=-0.291


...

--- step 3000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.016 m/s vel_tracking_reward=0.9974
  pitch=-0.028 roll=-0.009


New command: cmd_vel=0.000


New command: cmd_vel=0.524
New command: cmd_vel=0.000


Mean return: 11397.77


Exported charts_metrics.csv (5 metrics, 966 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 8: Mass + friction randomization

In [28]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-08"

dr.mass_scale_range = (0.8, 1.2)
dr.friction_scale_range = (0.5, 1.5)
for w in envs.envs:
    w.env.dr = dr


In [29]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-08__42__1788744232
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-08
New episode: cmd_vel=0.395


New command: cmd_vel=0.000


New command: cmd_vel=0.000


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.028 m/s vel_tracking_reward=0.9920
  pitch=-0.005 roll=-0.011
New command: cmd_vel=0.000


New command: cmd_vel=-0.388


...

New command: cmd_vel=-0.703


New command: cmd_vel=0.000


--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.017 m/s vel_tracking_reward=0.9970
  pitch=0.035 roll=0.011


New command: cmd_vel=0.000


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-08__42__1788744232/uniwheel-ppo-cmd-phase-08_final.cleanrl_model
Best model mean return: 17927.42, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-08__42__1788744232/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-08__42__1788744232/best_model.cleanrl_model
Best mean return: 17927.42


In [30]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=0.000


New command: cmd_vel=0.000


New command: cmd_vel=0.000
--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.006 m/s vel_tracking_reward=0.9996
  pitch=0.007 roll=0.018


New command: cmd_vel=0.482


New command: cmd_vel=-0.291


...

New command: cmd_vel=-0.606


--- step 4000 ---
  cmd_vel=-0.606 vel_target=-0.303 m/s vel_actual=-0.139 m/s vel_tracking_reward=0.7635
  pitch=-0.026 roll=0.006


New command: cmd_vel=-0.198


Mean return: 11625.46


Exported charts_metrics.csv (5 metrics, 893 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 9: Motor gain randomization

In [31]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-09"

dr.motor_gain_range = (0.6, 1.0)
for w in envs.envs:
    w.env.dr = dr


In [32]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-09__42__1788749197
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-09
New episode: cmd_vel=-0.812


New command: cmd_vel=-0.021


New command: cmd_vel=0.000


New command: cmd_vel=0.000


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.049 m/s vel_tracking_reward=0.9766
  pitch=0.008 roll=0.009
New command: cmd_vel=0.000


...

New command: cmd_vel=-0.703


New command: cmd_vel=0.000


--- step 4000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=0.015 m/s vel_tracking_reward=0.9977
  pitch=0.013 roll=-0.005


New command: cmd_vel=0.000


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-09__42__1788749197/uniwheel-ppo-cmd-phase-09_final.cleanrl_model
Best model mean return: 13737.59, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-09__42__1788749197/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-09__42__1788749197/best_model.cleanrl_model
Best mean return: 13737.59


In [33]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=0.346


New command: cmd_vel=0.000


New command: cmd_vel=0.000


--- step 1000 ---
  cmd_vel=0.000 vel_target=0.000 m/s vel_actual=-0.019 m/s vel_tracking_reward=0.9965
  pitch=0.020 roll=-0.002


New command: cmd_vel=0.482


...

New command: cmd_vel=-0.478


--- step 2000 ---
  cmd_vel=-0.478 vel_target=-0.239 m/s vel_actual=-0.095 m/s vel_tracking_reward=0.8119
  pitch=-0.022 roll=0.013


New command: cmd_vel=0.000


Mean return: 1348.14


Exported charts_metrics.csv (5 metrics, 1179 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 10: Ridge torque (wheel axle only)

Same physically-motivated restriction as stage 04 - not applied to `reaction_wheel`,
which never touches the ground (see `uniwheel_env_cmd.py`'s module docstring).

In [34]:
ppo_config.exp_name = "uniwheel-ppo-cmd-phase-10"

dr.ridge_prob = 0.05
dr.ridge_torque_max_nm = 0.005
for w in envs.envs:
    w.env.dr = dr
    w.env.smoothness_penalty_coef = 0.5


In [35]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


Run name: UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064
TensorBoard: http://localhost:6006/#scalars&regexFilter=uniwheel-ppo-cmd-phase-10
New episode: cmd_vel=-0.812


New command: cmd_vel=0.000


New command: cmd_vel=0.000


New command: cmd_vel=0.408


--- step 1000 ---
  cmd_vel=0.408 vel_target=0.204 m/s vel_actual=0.091 m/s vel_tracking_reward=0.8804
  pitch=0.017 roll=-0.000


...

New command: cmd_vel=0.000


New command: cmd_vel=0.448


New command: cmd_vel=0.000
New command: cmd_vel=0.351


New command: cmd_vel=0.000


Final model saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/uniwheel-ppo-cmd-phase-10_final.cleanrl_model
Best model mean return: 16254.31, saved to runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/best_model.cleanrl_model
Best model: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/best_model.cleanrl_model
Best mean return: 16254.31


In [36]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


New episode: cmd_vel=0.000
New command: cmd_vel=0.816


New command: cmd_vel=0.000


New command: cmd_vel=-0.598
New command: cmd_vel=-0.820


New command: cmd_vel=0.000


New command: cmd_vel=0.000


...

New command: cmd_vel=0.959
New command: cmd_vel=-0.170


New command: cmd_vel=0.000


New command: cmd_vel=0.000


Mean return: 2872.58


Exported charts_metrics.csv (5 metrics, 1087 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Clean up and export

In [37]:
for w in envs.envs:
    w.env.close()


In [38]:
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)
print(f"Exported {result.checkpoint_dir / 'actor.onnx'}")
print("Next: python3 ../rl/onnx_actor_to_c.py " + str(result.checkpoint_dir / "actor.onnx") +
      " actor.h, then copy actor.h into ../03-sim-to-real/uniwheel_bot/")


/home/jainajeet29/Uni Wheel/workspace/software/05-command-conditioning/../rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


W0907 10:58:49.442000 22515 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms


W0907 10:58:49.443000 22515 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align


W0907 10:58:49.444000 22515 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool


W0907 10:58:49.445000 22515 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/actor.onnx
Exported runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/actor.onnx
Next: python3 ../rl/onnx_actor_to_c.py runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/actor.onnx actor.h, then copy actor.h into ../03-sim-to-real/uniwheel_bot/
